# 01 — Data Extraction

## 1. Introduction

This is the first of five notebooks implementing *"Mapping the Gap: Health
& Education Accessibility in Akure North and Akure South"* — a submission
to **Map<>kathon 2026** (OSM Dashboard or Analysis track).

### What this notebook does

It pulls the raw geographic data this entire project is built on, straight
from OpenStreetMap (OSM) — the free, community-editable map of the world.
Specifically, for each of our two study areas (Akure North and Akure South
Local Government Areas, in Ondo State, Nigeria), it retrieves:

- **Roads** — the street network, used later for travel-time routing
- **Buildings** — used as a proxy for where people actually live
- **Health facilities** and **schools** — the services we're measuring
  accessibility *to*
- **Waterways** and **land use** — supporting context layers

None of this extraction logic lives in this notebook itself: it's all
implemented, tested, and documented in a separate companion tool,
**`lga_extractor`** (from the sibling `lga-osm-extractor` repository,
submitted separately as its own Lightweight Tool/Demo entry). This
notebook's job is simply to *call* that tool for our two specific study
areas and confirm the output looks right before the rest of the pipeline
depends on it.

### Why start here?

Every later notebook — the completeness check (02), the accessibility
scoring (03), the summary tables (04), and the interactive maps (05) — all
depend on the files this notebook produces. If something goes wrong here
(a boundary resolves incorrectly, a layer comes back empty), it will
silently propagate through every subsequent step. That's why Section 5
below is dedicated entirely to *validating* what came out, not just
running the extraction and moving on.

### Notebook pipeline (this project)

| # | Notebook | Purpose |
|---|---|---|
| **01** | **Data Extraction** | **← you are here** |
| 02 | Completeness Assessment | Flag likely OSM data gaps |
| 03 | Accessibility Analysis | Travel-time scoring (walk / okada / drive) |
| 04 | Results Summary | Combine both LGAs, rank underserved areas |
| 05 | Kepler.gl Visualization | Polished exportable map visuals |

### Expected outputs

For each study area, a folder of GeoJSON/Shapefile layers plus a
`run_log.json` recording exactly what was queried and when (see Section
7, Export).

### Prerequisites

- The `lga_extractor` package, either installed in editable mode
  (`pip install -e ../../lga-osm-extractor`) or importable via `sys.path`.
- Internet access — this notebook makes **live queries to OSM's Overpass
  API**, so it will not run offline, and results can vary slightly over
  time as OSM data is continuously edited by contributors worldwide.

## 2. Imports

This notebook only needs two things imported directly: the companion
extraction tool, and (later, for validation) `geopandas`/`pandas` to
inspect what came back. Everything else — the actual OSM querying,
cleaning, and file-writing — happens inside `lga_extractor`, which is why
this notebook's own import list is short.

### 2.1 Environment setup

Before we can `import lga_extractor`, Python needs to be able to find it.
This differs between Google Colab (where the repo lives in Google Drive)
and a local machine (where it's a sibling folder) — the cell below detects
which environment we're in and adjusts accordingly.

Detects Colab vs. local automatically. In Colab, this assumes you've
created a Google Drive folder named **`Akure Access Dashboard`** at the
root of My Drive containing this repository's contents, and — for the
`lga_extractor` import to work — that the companion tool's repository is
either placed alongside it in Drive (as **`LGA OSM Extractor`**) or
installed separately.

In [ ]:
# --- Environment setup ---
import sys, os

IN_COLAB = "google.colab" in sys.modules

DASHBOARD_DRIVE_FOLDER = "Akure Access Dashboard"
EXTRACTOR_DRIVE_FOLDER = "LGA OSM Extractor"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DASHBOARD_DIR = f"/content/drive/MyDrive/{DASHBOARD_DRIVE_FOLDER}"
    EXTRACTOR_DIR = f"/content/drive/MyDrive/{EXTRACTOR_DRIVE_FOLDER}"

    if not os.path.exists(DASHBOARD_DIR):
        raise FileNotFoundError(
            f"Expected '{DASHBOARD_DIR}' in your Google Drive.\n"
            f"Create a folder named '{DASHBOARD_DRIVE_FOLDER}' at the root "
            f"of My Drive and upload this repository into it."
        )
    if not os.path.exists(EXTRACTOR_DIR):
        print(
            f"Warning: '{EXTRACTOR_DIR}' not found in Drive. The "
            f"lga_extractor import below will fail unless it's installed "
            f"another way (e.g. 'pip install lga-extractor' if published, "
            f"or adjust sys.path manually)."
        )

    %cd {DASHBOARD_DIR}/notebooks
    !pip install osmnx geopandas shapely fiona pandas leafmap keplergl --quiet

    if os.path.exists(EXTRACTOR_DIR):
        sys.path.append(EXTRACTOR_DIR)

    print(f"Running in Colab. Working directory: {os.getcwd()}")
else:
    sys.path.append("../../lga-osm-extractor")  # adjust path as needed locally
    print("Running locally. Assuming dependencies are installed and the "
          "lga-osm-extractor repo is a sibling directory (adjust the path "
          "above if your layout differs).")


### 2.2 Package import

With the path set up, we can now import the one function this notebook
actually calls: `extract_lga`. This single function wraps the entire
extraction pipeline (boundary resolution → OSM querying → cleaning →
export → logging) documented in `lga_extractor/README.md`.

In [ ]:
from lga_extractor import extract_lga

## 3. Configuration

Both study areas and their output paths are defined here in one place,
rather than scattered through the notebook, so the whole extraction run
can be re-parameterized (e.g. to add a third LGA) by editing just this
one cell.

- **`lga_name`** / **`state_name`** — used by `lga_extractor` to resolve
  each LGA's boundary via OSM's own place-name data (see Notebook 02's
  discussion of boundary resolution for more on how this works and what
  can go wrong).
- **`output_dir`** — where this LGA's extracted layers get written; every
  downstream notebook reads from these same paths.

In [ ]:
STUDY_AREAS = [
    {"lga_name": "Akure North", "state_name": "Ondo", "output_dir": "../data/processed/akure_north"},
    {"lga_name": "Akure South", "state_name": "Ondo", "output_dir": "../data/processed/akure_south"},
]

for area in STUDY_AREAS:
    print(f"{area['lga_name']}, {area['state_name']} -> {area['output_dir']}")


## 4. Data Loading

Unlike later notebooks, this one doesn't *load* pre-existing data — it
*generates* the data that later notebooks will load. Conceptually this
section plays the same role a "Data Loading" section would in a notebook
further down the pipeline: after this section, you have the raw material
everything else depends on.

### 4.1 Extraction

Runs the full `lga_extractor` pipeline for each study area in turn. Under
the hood, for each LGA, this:

1. **Resolves the boundary** — turns "Akure North, Ondo" into an actual
   polygon geometry, by querying OSM's place/administrative-boundary data
   (this is what geocoding means here: not converting an address to a
   point, but resolving a place *name* to its full boundary *shape*).
2. **Queries OSM's Overpass API** for each configured feature type (roads,
   buildings, etc.) within that boundary, using OSM's tagging system —
   e.g. `highway=*` for any road, `amenity=hospital` or `amenity=clinic`
   for health facilities.
3. **Cleans and standardizes** the results: reprojects to a consistent
   metric coordinate reference system (EPSG:32631, UTM Zone 31N — correct
   for this part of Nigeria), removes duplicate features, and
   standardizes column names across layers.
4. **Exports** each layer as GeoJSON (and Shapefile, where geometry types
   allow — see the extractor's own documentation for why some layers get
   split into multiple Shapefiles).
5. **Logs the run** — records exactly what was queried, when, and with
   which package versions, so results can be traced and reproduced later.

See the companion tool's own example notebook
(`lga-osm-extractor/examples/extract_akure_lgas.ipynb`) for a more
detailed, step-by-step walkthrough of what happens inside this call.

In [ ]:
results = {}

for area in STUDY_AREAS:
    print(f"Extracting {area['lga_name']}...")
    result = extract_lga(
        lga_name=area["lga_name"],
        state_name=area["state_name"],
        output_dir=area["output_dir"],
    )
    results[area["lga_name"]] = result
    print(f"  Boundary source: {result['boundary_source']}")
    print(f"  Output: {result['output_dir']}")
    if result["warnings"]:
        for w in result["warnings"]:
            print(f"  Warning: {w}")
    print()


## 5. Processing

### 5.1 Validation

Confirms every expected layer was extracted with a non-trivial feature
count, for both study areas, before downstream notebooks depend on this
data. This is a deliberately simple sanity check — not a full data-quality
audit — but it catches the most common failure mode: a layer coming back
empty because of a tag misconfiguration, an Overpass timeout, or (rarely)
a genuinely empty result.

In [ ]:
import geopandas as gpd
import pandas as pd

summary_rows = []
for lga_name, result in results.items():
    row = {"LGA": lga_name}
    for layer_name, paths in result["exported"].items():
        if layer_name.startswith("_"):  # skip metadata keys like _skipped, _split_layers
            continue
        gdf = gpd.read_file(paths["geojson"])
        row[layer_name] = len(gdf)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("LGA")
summary_df


If any expected layer shows **0** above, check the corresponding run's
warnings in Section 4.1 output — this usually means either the area
genuinely has no OSM features of that type yet (a completeness signal in
its own right, explored in Notebook 02), or the boundary resolved
incorrectly and should be checked manually.

## 6. Visualization

This notebook doesn't produce a map of its own — with only tabular
feature counts to show, a visual here wouldn't add much beyond the table
above. If you want a quick visual check of what was actually extracted
(rather than just counts), the companion extractor tool has an optional
`--preview` flag that renders a kepler.gl map of the raw layers; the full
polished, publication-ready visuals for the *scored* accessibility data
come later, in Notebook 05.

## 7. Export

**Outputs produced (per LGA):**

```
data/processed/{lga_name}/
├── roads.geojson (+ shapefiles/roads.shp)
├── buildings.geojson
├── waterways.geojson
├── landuse.geojson
├── health_facilities.geojson
├── schools.geojson
└── run_log.json
```

These files are written directly by `lga_extractor`'s `export_layers()`
and `log_run()` functions (Section 4.1) — nothing further needs to be
saved from this notebook itself. `run_log.json` in particular is worth
keeping: it records the exact package versions and query configuration
used, which is what makes this extraction reproducible later (see the
root README's reproducibility notes).

## 8. Summary

**What this notebook accomplished:** extracted six OSM feature layers for
both Akure North and Akure South, validated that every layer came back
with real data, and logged the extraction for reproducibility.

**Outputs produced:** see Section 7 above — one folder of GeoJSON/Shapefile
layers plus a run log, per LGA.

**Next notebook:** `02_completeness_assessment.ipynb` uses the buildings,
health facility, and school layers produced here to build the analysis
grid and flag cells that appear settled but lack a nearby OSM facility
tag — a likely data gap rather than a confirmed service gap.